## 1. Imports

NumPy est utilisé pour manipuler efficacement des tableaux de données, `time` pour mesurer les temps d'exécution et Matplotlib pour visualiser les résultats.

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt

print("Bibliothèques chargées avec succès.")

Matplotlib is building the font cache; this may take a moment.


Bibliothèques chargées avec succès.


## 2. Multiplication scalaire

On commence par l'opération la plus simple : la multiplication de deux entiers. Cette fonction servira de référence fonctionnelle pour la future IP HLS.

In [2]:
def multiplier(a, b):
    """Retourne le produit de deux entiers."""
    return a * b

a = 12
b = 7
result = multiplier(a, b)

print(f"{a} x {b} = {result}")

12 x 7 = 84


## 3. Tests fonctionnels

Avant toute implémentation matérielle, il est important de vérifier le comportement de la fonction sur plusieurs cas : valeurs positives, zéro et valeurs négatives.

In [3]:
test_cases = [
    (12, 7, 84),
    (0, 25, 0),
    (9, 9, 81),
    (-4, 6, -24),
    (-5, -8, 40),
]

all_ok = True

for a, b, expected in test_cases:
    obtained = multiplier(a, b)
    ok = obtained == expected
    all_ok &= ok
    print(f"{a:4d} x {b:4d} = {obtained:6d} | attendu = {expected:6d} | {'PASS' if ok else 'FAIL'}")

print("\nValidation globale :", "PASS" if all_ok else "FAIL")

  12 x    7 =     84 | attendu =     84 | PASS
   0 x   25 =      0 | attendu =      0 | PASS
   9 x    9 =     81 | attendu =     81 | PASS
  -4 x    6 =    -24 | attendu =    -24 | PASS
  -5 x   -8 =     40 | attendu =     40 | PASS

Validation globale : PASS


## 4. Multiplication de vecteurs

La version DMA du tutoriel devra traiter un volume de données plus important. On introduit donc dès maintenant la multiplication élément par élément de deux vecteurs :

`C[i] = A[i] × B[i]`

Cette opération sera conservée dans la version FPGA afin de comparer les résultats.

In [4]:
A = np.array([1, 2, 3, 4, 5], dtype=np.int32)
B = np.array([10, 20, 30, 40, 50], dtype=np.int32)

C = A * B

print("A =", A)
print("B =", B)
print("C =", C)

A = [1 2 3 4 5]
B = [10 20 30 40 50]
C = [ 10  40  90 160 250]


## 5. Vérification automatique sur un jeu de données aléatoire

On génère maintenant deux vecteurs aléatoires. Le résultat obtenu servira de **golden reference** pour vérifier les futures sorties FPGA.

In [5]:
rng = np.random.default_rng(seed=42)

N = 1024
A = rng.integers(-100, 101, size=N, dtype=np.int32)
B = rng.integers(-100, 101, size=N, dtype=np.int32)

reference = A * B

print("Nombre d'éléments :", N)
print("Premières valeurs de A :", A[:10])
print("Premières valeurs de B :", B[:10])
print("Premiers résultats      :", reference[:10])

Nombre d'éléments : 1024
Premières valeurs de A : [-83  55  31 -12 -13  72 -83  40 -60 -82]
Premières valeurs de B : [ 65 -89 -85 -44  13 -33  -3 -66  46 -37]
Premiers résultats      : [-5395 -4895 -2635   528  -169 -2376   249 -2640 -2760  3034]


## 6. Implémentation CPU avec une boucle Python

Cette version traite les données une par une. Elle est volontairement simple et permet d'obtenir une première référence de temps d'exécution.

In [6]:
def multiply_python_loop(a, b):
    result = np.empty(len(a), dtype=np.int32)
    for i in range(len(a)):
        result[i] = int(a[i]) * int(b[i])
    return result

result_loop = multiply_python_loop(A, B)

print("Résultat correct :", np.array_equal(result_loop, reference))

Résultat correct : True


## 7. Implémentation NumPy

NumPy exploite des routines optimisées pour effectuer directement les opérations sur les tableaux. Cette version représente une référence logicielle plus performante sur CPU.

In [7]:
def multiply_numpy(a, b):
    return a * b

result_numpy = multiply_numpy(A, B)

print("Résultat correct :", np.array_equal(result_numpy, reference))

Résultat correct : True
